# 🛍️ Laboratorio: DBSCAN con el caso Mall Customers

## 📖 La historia
En la sesión de K-Means segmentamos a los **200 clientes** del centro comercial en **6 grupos** con su edad, ingreso anual y gasto, y le propusimos una campaña a cada uno. Marketing quedó contento, pero el gerente de lealtad levantó la mano:

> *"K-Means metió a **todos** los clientes en algún segmento. ¿De verdad todos se parecen a su grupo? Yo conozco clientes que no se parecen a nadie, y no quiero mandarles la campaña equivocada."*

K-Means **no puede** responder eso: por diseño asigna cada punto al centroide más cercano, aunque esté lejísimos. DBSCAN sí: lo que no está en una zona densa lo marca como **ruido**.

> ### ❓ ¿Qué clientes no encajan en ningún segmento, quiénes son y qué hacemos con ellos?

## 🧭 Lo que vas a aprender
1. Elegir `min_samples` con la regla práctica y `eps` con la **gráfica de k-distancia**
2. Buscar parámetros con una **grilla** y leer el **trade-off** entre silhouette y % de ruido
3. **Perfilar el ruido**: ¿son errores, atípicos interesantes o un segmento disperso?
4. Comparar DBSCAN con K-Means de forma **justa** (mismo conjunto de clientes) y con el **ARI** entre ambos
5. Traducir el resultado a una **recomendación** de negocio

## ⏱️ Agenda (primera mitad de la sesión: 90 min)
| Parte | Tema | Tiempo |
|---|---|---|
| 0 | Preparación y recap de K-Means | 5 min |
| 1 | `min_samples` y la gráfica de k-distancia | 15 min |
| 2 | Grilla `eps` × `min_samples`: el trade-off del ruido | 20 min |
| 3 | El modelo elegido | 10 min |
| 4 | ¿Quiénes son el ruido? | 15 min |
| 5 | DBSCAN vs K-Means | 10 min |
| 6 | Conclusiones | 5 min |
| ☕ | Descanso (después seguimos con `caso_entregas.ipynb`) | 10 min |

## 📋 Las variables
| Variable | Descripción |
|---|---|
| `cliente_id` | Identificador del cliente (no se usa para agrupar) |
| `genero` | `Male` / `Female` (no se usa para agrupar) |
| `edad` | Edad en años |
| `ingreso_anual` | Ingreso anual en **miles de dólares** (k$) |
| `gasto` | *Spending score* de 1 a 100 que asigna el centro comercial (más alto = gasta más) |

---
# 0️⃣ Preparación (5 min)

Ejecuta las siguientes celdas **una sola vez**.

## 🛠️ Funciones auxiliares de visualización

Ejecuta la siguiente celda **una sola vez** al inicio. Después sólo tienes que **llamar** a la función que necesites:

| Función | ¿Para qué sirve? |
|---|---|
| `plot_distributions(df, columnas)` | Histograma + boxplot de variables numéricas |
| `plot_correlation_matrix(df, columnas)` | Matriz de correlación |
| `plot_pairplot(df, columnas, color=None)` | Dispersión entre todas las variables numéricas |
| `plot_clusters(df, x, y, labels, centroides=None)` | Puntos en 2D coloreados por cluster (el ruido en gris) |
| `plot_clusters_3d(df, x, y, z, labels)` | Lo mismo en 3D (se puede rotar) |
| `plot_codo(ks, inercias)` | Método del codo para elegir k |
| `plot_metricas_k(tabla)` | Silhouette, Davies-Bouldin y Calinski-Harabasz según k |
| `plot_silueta(X, labels)` | Coeficiente de silueta de cada punto, por cluster |
| `plot_k_distancia(X, k, eps=None)` | Distancia al k-ésimo vecino: ayuda a elegir `eps` de DBSCAN |
| `plot_varianza_explicada(pca)` | Varianza explicada por componente y acumulada |
| `plot_cargas(pca, columnas)` | Peso de cada variable original en cada componente |
| `plot_biplot(scores, pca, columnas, labels=None)` | Puntos en PC1-PC2 con una flecha por variable |

In [ ]:
# Funciones auxiliares de visualización
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _etiquetas_texto(labels):
    """Convierte etiquetas de cluster a texto para que plotly use colores discretos; -1 = 'Ruido'."""
    labels = np.asarray(labels)
    return np.where(labels == -1, 'Ruido', np.char.add('Cluster ', labels.astype(str)))


def _orden_clusters(texto):
    """Orden de la leyenda: Cluster 0, 1, 2, … y 'Ruido' al final."""
    unicos = sorted(set(texto), key=lambda t: (t == 'Ruido', int(t.split()[-1]) if t != 'Ruido' else 0))
    return {'cluster': unicos}


def plot_distributions(df, columns, nbins=30):
    """Histograma con boxplot marginal para cada variable numérica."""
    for col in columns:
        fig = px.histogram(
            df,
            x=col,
            nbins=nbins,
            marginal='box',
            opacity=0.7,
            title=f'Distribución de {col}'
        )
        fig.update_layout(bargap=0.2)
        fig.show()


def plot_correlation_matrix(df, columns):
    """Mapa de calor con la correlación de Pearson entre las variables numéricas."""
    corr = df[columns].corr().round(2)
    fig = px.imshow(
        corr,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        title='Matriz de Correlación'
    )
    fig.update_layout(width=750, height=650)
    fig.show()


def plot_pairplot(df, columns, color=None):
    """Matriz de dispersión (pairplot) entre las variables numéricas."""
    fig = px.scatter_matrix(
        df,
        dimensions=columns,
        color=color,
        title='Pairplot de Variables Numéricas'
    )
    fig.update_layout(width=1000, height=1000, title_font_size=20)
    fig.update_traces(diagonal_visible=True, marker={'size': 4, 'opacity': 0.6})
    fig.show()


def plot_clusters(df, x, y, labels, centroides=None, title='Clusters'):
    """Dispersión 2D coloreada por cluster. El ruido de DBSCAN (-1) aparece en gris.
    centroides: arreglo (k, 2) en las mismas unidades que x, y (opcional)."""
    datos = df[[x, y]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter(
        datos,
        x=x,
        y=y,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    if centroides is not None:
        centroides = np.asarray(centroides)
        fig.add_trace(go.Scatter(
            x=centroides[:, 0],
            y=centroides[:, 1],
            mode='markers',
            marker={'symbol': 'x', 'size': 16, 'color': 'black', 'line': {'width': 2}},
            name='Centroides'
        ))
    fig.update_layout(width=800, height=550)
    fig.show()


def plot_clusters_3d(df, x, y, z, labels, title='Clusters en 3D'):
    """Dispersión 3D coloreada por cluster (se puede rotar con el mouse)."""
    datos = df[[x, y, z]].copy()
    datos['cluster'] = _etiquetas_texto(labels)
    fig = px.scatter_3d(
        datos,
        x=x,
        y=y,
        z=z,
        color='cluster',
        category_orders=_orden_clusters(datos['cluster']),
        color_discrete_map={'Ruido': 'lightgray'},
        opacity=0.8,
        title=title
    )
    fig.update_traces(marker={'size': 4})
    fig.update_layout(width=850, height=650)
    fig.show()


def plot_codo(ks, inercias):
    """Método del codo: inercia (suma de distancias² al centroide) para cada k."""
    fig = px.line(
        x=list(ks),
        y=list(inercias),
        markers=True,
        labels={'x': 'Número de clusters (k)', 'y': 'Inercia'},
        title='Método del codo'
    )
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_metricas_k(tabla):
    """Silhouette, Davies-Bouldin y Calinski-Harabasz para cada k (tabla de tabla_metricas_k)."""
    metricas = [
        ('silhouette', 'Silhouette (↑ mejor)'),
        ('davies_bouldin', 'Davies-Bouldin (↓ mejor)'),
        ('calinski_harabasz', 'Calinski-Harabasz (↑ mejor)'),
    ]
    fig = make_subplots(rows=1, cols=3, subplot_titles=[t for _, t in metricas])
    for i, (col, _) in enumerate(metricas, start=1):
        fig.add_trace(go.Scatter(x=tabla['k'], y=tabla[col], mode='lines+markers', name=col), row=1, col=i)
        fig.update_xaxes(title_text='k', dtick=1, row=1, col=i)
    fig.update_layout(width=1150, height=400, showlegend=False, title='Métricas internas según k')
    fig.show()


def plot_silueta(X, labels, title=None):
    """Gráfica de silueta: el ancho de cada barra es el coeficiente de un punto, agrupado por cluster.
    La línea punteada es el promedio. Los puntos de ruido (-1) se excluyen."""
    from sklearn.metrics import silhouette_samples, silhouette_score

    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    X, labels = X[mask], labels[mask]
    valores = silhouette_samples(X, labels)
    promedio = silhouette_score(X, labels)

    fig = go.Figure()
    y_inf = 10
    for c in np.unique(labels):
        v = np.sort(valores[labels == c])
        y = np.arange(y_inf, y_inf + len(v))
        fig.add_trace(go.Scatter(
            x=v, y=y, fill='tozerox', mode='lines', line={'width': 0.5}, name=f'Cluster {c}'
        ))
        y_inf += len(v) + 10
    fig.add_vline(x=promedio, line_dash='dash', line_color='red',
                  annotation_text=f'promedio = {promedio:.3f}', annotation_position='top')
    fig.update_layout(
        title=title or f'Gráfica de silueta (k = {len(np.unique(labels))})',
        xaxis_title='Coeficiente de silueta',
        yaxis={'showticklabels': False, 'title': 'Puntos (ordenados por cluster)'},
        width=750,
        height=500
    )
    fig.show()


def plot_k_distancia(X, k, eps=None):
    """Distancia que necesita cada punto para tener k vecinos (contándose a sí mismo), ordenada.
    Usa k = min_samples: igual que DBSCAN, el punto cuenta como uno de sus k vecinos, así que un punto
    es núcleo con un eps dado si su distancia en esta curva es <= eps. El 'codo' sugiere un buen eps.
    eps (opcional) dibuja una línea horizontal de referencia."""
    from sklearn.neighbors import NearestNeighbors

    # n_neighbors=k incluye al propio punto (distancia 0), como min_samples en DBSCAN
    vecinos = NearestNeighbors(n_neighbors=k).fit(X)
    distancias, _ = vecinos.kneighbors(X)
    d_k = np.sort(distancias[:, -1])

    fig = px.line(
        x=np.arange(len(d_k)),
        y=d_k,
        labels={'x': 'Puntos ordenados por distancia', 'y': f'Distancia para tener {k} vecinos (incluido él)'},
        title=f'Gráfica de k-distancia (k = {k})'
    )
    if eps is not None:
        fig.add_hline(y=eps, line_dash='dash', line_color='red',
                      annotation_text=f'eps = {eps}', annotation_position='top left')
    fig.update_layout(width=750, height=450)
    fig.show()


def plot_varianza_explicada(pca):
    """Scree plot: varianza explicada por componente (barras) y acumulada (línea)."""
    ratio = pca.explained_variance_ratio_
    n = np.arange(1, len(ratio) + 1)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=n, y=ratio, name='Por componente'))
    fig.add_trace(go.Scatter(x=n, y=np.cumsum(ratio), mode='lines+markers', name='Acumulada'))
    for umbral in (0.80, 0.95):
        fig.add_hline(y=umbral, line_dash='dot', line_color='gray',
                      annotation_text=f'{umbral:.0%}', annotation_position='right')
    fig.update_layout(
        title='Varianza explicada por componente',
        xaxis={'title': 'Componente principal', 'dtick': 1 if len(ratio) <= 30 else None},
        yaxis={'title': 'Proporción de varianza', 'tickformat': '.0%', 'range': [0, 1.05]},
        width=800,
        height=450
    )
    fig.show()


def plot_cargas(pca, columnas, n_componentes=None):
    """Mapa de calor de las cargas (loadings): cuánto pesa cada variable original en cada componente."""
    n = n_componentes or pca.n_components_
    cargas = pd.DataFrame(
        pca.components_[:n].T,
        index=columnas,
        columns=[f'PC{i + 1}' for i in range(n)]
    ).round(2)
    fig = px.imshow(
        cargas,
        text_auto=True,
        color_continuous_scale='RdBu_r',
        zmin=-1,
        zmax=1,
        aspect='auto',
        title='Cargas de cada variable en los componentes'
    )
    fig.update_layout(width=200 + 90 * n, height=150 + 40 * len(columnas))
    fig.show()


def plot_biplot(scores, pca, columnas, labels=None, escala=None, title='Biplot PC1 vs PC2'):
    """Puntos en el plano PC1-PC2 más una flecha por variable original (sus cargas).
    scores: salida de pca.transform(X). labels (opcional) colorea los puntos por cluster o grupo."""
    scores = np.asarray(scores)
    datos = pd.DataFrame({'PC1': scores[:, 0], 'PC2': scores[:, 1]})
    if labels is not None:
        datos['grupo'] = _etiquetas_texto(labels) if np.issubdtype(np.asarray(labels).dtype, np.integer) \
            else np.asarray(labels).astype(str)
    fig = px.scatter(datos, x='PC1', y='PC2', color='grupo' if labels is not None else None,
                     opacity=0.6, title=title)
    cargas = pca.components_[:2].T
    escala = escala or 0.9 * np.abs(scores[:, :2]).max() / np.abs(cargas).max()
    for (cx, cy), nombre in zip(cargas * escala, columnas):
        fig.add_annotation(x=cx, y=cy, ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
                           showarrow=True, arrowhead=2, arrowwidth=1.5, arrowcolor='black')
        fig.add_annotation(x=cx * 1.08, y=cy * 1.08, text=nombre, showarrow=False, font={'size': 12})
    r = pca.explained_variance_ratio_
    fig.update_layout(
        xaxis_title=f'PC1 ({r[0]:.1%} de la varianza)',
        yaxis_title=f'PC2 ({r[1]:.1%} de la varianza)',
        width=850,
        height=650
    )
    fig.show()

### 🧰 Funciones de clustering

Hacen en una línea lo que de otra forma repetirías con cada modelo:

| Función | ¿Para qué sirve? |
|---|---|
| `evaluar_clustering(nombre, X, labels)` | Número de clusters, % de ruido, silhouette, Davies-Bouldin y Calinski-Harabasz (sin contar el ruido) |
| `tabla_metricas_k(X, ks)` | Ajusta K-Means para cada k y regresa inercia y métricas en una tabla |
| `perfil_clusters(df, labels, columnas)` | Tamaño y promedio de cada variable por cluster |

In [ ]:
# Funciones de clustering
# Ejecuta esta celda una vez; después sólo llama a las funciones.
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def evaluar_clustering(nombre, X, labels):
    """Métricas internas de un agrupamiento. El ruido (-1) se excluye de las métricas,
    pero se reporta qué porcentaje de los puntos quedó como ruido."""
    X = np.asarray(X)
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(np.unique(labels[mask]))
    fila = {
        'modelo': nombre,
        'n_clusters': n_clusters,
        'pct_ruido': float(round(100 * (~mask).mean(), 1)),
    }
    if n_clusters >= 2:
        fila['silhouette'] = float(round(silhouette_score(X[mask], labels[mask]), 3))
        fila['davies_bouldin'] = float(round(davies_bouldin_score(X[mask], labels[mask]), 3))
        fila['calinski_harabasz'] = float(round(calinski_harabasz_score(X[mask], labels[mask]), 1))
    else:
        fila.update({'silhouette': np.nan, 'davies_bouldin': np.nan, 'calinski_harabasz': np.nan})
    return fila


def tabla_metricas_k(X, ks, random_state=42):
    """Ajusta K-Means para cada k y regresa una tabla con inercia, silhouette, Davies-Bouldin y Calinski-Harabasz."""
    filas = []
    for k in ks:
        modelo = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
        fila = evaluar_clustering(f'k={k}', X, modelo.labels_)
        filas.append({'k': k, 'inercia': float(round(modelo.inertia_, 1)), **{
            m: fila[m] for m in ('silhouette', 'davies_bouldin', 'calinski_harabasz')}})
    return pd.DataFrame(filas)


def perfil_clusters(df, labels, columnas):
    """Promedio de cada variable por cluster, con tamaño y porcentaje. Sirve para ponerle nombre a cada grupo."""
    datos = df[columnas].copy()
    datos['cluster'] = np.asarray(labels)
    perfil = datos.groupby('cluster')[columnas].mean().round(1)
    perfil.insert(0, 'n', datos['cluster'].value_counts().sort_index())
    perfil.insert(1, 'pct', (100 * perfil['n'] / perfil['n'].sum()).round(1))
    perfil.index = ['Ruido' if c == -1 else f'Cluster {c}' for c in perfil.index]
    return perfil

### 📥 Los datos, ya preparados

Este caso lo trabajaste en la sesión de K-Means, así que no repetimos la exploración: la celda siguiente trae las dos funciones que hacen lo mismo que hiciste a mano.

| Función | ¿Qué hace? |
|---|---|
| `cargar_mall()` | Descarga los 200 clientes y renombra las columnas al español (`edad`, `ingreso_anual`, `gasto`) |
| `preparar_mall(df, columnas)` | Estandariza las columnas elegidas y regresa `X_scaled` y el `scaler` |

Usa exactamente el **mismo escalado** de aquella sesión, así que las métricas son comparables.

In [ ]:
# Datos del caso Mall Customers
import pandas as pd
from sklearn.preprocessing import StandardScaler

URL_MALL = 'https://raw.githubusercontent.com/jorgermzg15/data-files/refs/heads/main/Mall_Customers.csv'
COLS_MALL = ['edad', 'ingreso_anual', 'gasto']


def cargar_mall():
    """Descarga los 200 clientes del centro comercial y renombra las columnas al español."""
    df = pd.read_csv(URL_MALL)
    return df.rename(columns={
        'CustomerID': 'cliente_id',
        'Genre': 'genero',
        'Age': 'edad',
        'Annual Income (k$)': 'ingreso_anual',
        'Spending Score (1-100)': 'gasto',
    })


def preparar_mall(df, columnas=COLS_MALL):
    """Estandariza las columnas elegidas (media 0, desviación 1) y regresa la matriz escalada y el scaler."""
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df[columnas])
    return X_scaled, scaler

In [ ]:
df = cargar_mall()
X_scaled, scaler = preparar_mall(df)       # edad, ingreso_anual y gasto estandarizados

print(df.shape, X_scaled.shape)
df.head()

### 🧩 Funciones del caso
Una función que no está en los helpers del curso, porque sólo la usamos aquí:

| Función | ¿Para qué sirve? |
|---|---|
| `grilla_dbscan(X, eps_values, min_samples_values)` | Ajusta un DBSCAN por combinación y regresa una tabla con `evaluar_clustering` de cada uno |

In [ ]:
# Funciones del caso
from sklearn.cluster import DBSCAN, KMeans


def grilla_dbscan(X, eps_values, min_samples_values):
    """Ajusta DBSCAN para cada combinación de eps y min_samples.
    Regresa una tabla con el número de clusters, % de ruido y métricas (sin contar el ruido)."""
    filas = []
    for eps in eps_values:
        for ms in min_samples_values:
            labels = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
            fila = evaluar_clustering(f'eps={eps}, ms={ms}', X, labels)
            filas.append({'eps': eps, 'min_samples': ms, **fila})
    return pd.DataFrame(filas).drop(columns='modelo')

### 📋 Recap: lo que decidimos en la sesión de K-Means
- Variables: `edad`, `ingreso_anual` y `gasto`, **estandarizadas** (`preparar_mall`).
- Modelo: **K-Means con k = 6**, `n_init=10`, `random_state=42` → silhouette **0.428**, Davies-Bouldin **0.825**, Calinski-Harabasz **135.1**.
- Cada cliente quedó en uno de 6 segmentos, a los que les pusimos nombre:

| Cluster | Segmento | Clientes |
|---|---|---|
| 0 | Estándar maduro | 45 |
| 1 | Estándar joven | 39 |
| 2 | Cautelosos con dinero | 33 |
| 3 | VIP | 39 |
| 4 | Jóvenes impulsivos | 23 |
| 5 | Austeros | 21 |

La celda reproduce **exactamente** ese modelo para compararlo con DBSCAN al final.

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
kmeans_3d = KMeans(n_clusters=6, n_init=10, random_state=42).fit(X_scaled)

comparacion = []
comparacion.append(evaluar_clustering('K-Means (k=6)', X_scaled, kmeans_3d.labels_))

NOMBRES_KMEANS = {
    0: 'Estándar maduro',
    1: 'Estándar joven',
    2: 'Cautelosos con dinero',
    3: 'VIP',
    4: 'Jóvenes impulsivos',
    5: 'Austeros',
}
segmento_kmeans = pd.Series(kmeans_3d.labels_).map(NOMBRES_KMEANS).to_numpy()

perfil_kmeans = perfil_clusters(df, kmeans_3d.labels_, COLS_MALL)
perfil_kmeans.insert(0, 'segmento', list(NOMBRES_KMEANS.values()))
perfil_kmeans

💡 Fíjate en los dos segmentos de **gasto bajo**: **Cautelosos con dinero** (ingreso alto, gasto 17) y **Austeros** (ingreso bajo, gasto 19). Volveremos a ellos.

---
# 1️⃣ `min_samples` y la gráfica de k-distancia (15 min)

### Paso 1: fijar `min_samples`
Regla práctica: $\text{min\_samples} \ge \text{dimensiones} + 1$, y un buen arranque es $2 \times \text{dimensiones}$.

Tenemos **3 dimensiones** (`edad`, `ingreso_anual`, `gasto`) → mínimo **4**, arranque **6**.

### Paso 2: la gráfica de k-distancia
Con $k$ = `min_samples` = 6, calcula para cada cliente el radio que necesita para juntar 6 puntos **contándose a sí mismo** (igual que cuenta DBSCAN) y ordénalos. Ese radio es justo el `eps` mínimo con el que ese cliente sería núcleo. Los clientes "en grupo" tienen distancias chicas; los aislados, grandes.

In [ ]:
# Gráfica de k-distancia con k = 6
# Pista: plot_k_distancia(X_scaled, k=...)
# ✍️ Tu código aquí


Para leerla mejor, contamos qué % de clientes queda **arriba** de cada `eps` candidato. Un cliente arriba de la línea **no** junta 6 puntos (contándose a sí mismo) en ese radio, así que **no es núcleo** (será frontera o ruido):

In [ ]:
# Esta celda va de regalo: léela con calma y ejecútala
from sklearn.neighbors import NearestNeighbors

vecinos = NearestNeighbors(n_neighbors=6).fit(X_scaled)   # el 1er "vecino" es él mismo, como en min_samples
distancia_6 = vecinos.kneighbors(X_scaled)[0][:, -1]

pd.DataFrame({
    'eps': [0.4, 0.5, 0.6, 0.7, 0.8],
    'pct_no_nucleo': [round(100 * (distancia_6 > e).mean(), 1) for e in [0.4, 0.5, 0.6, 0.7, 0.8]]
})

✍️ **¿Qué observas?**
- ¿La curva tiene un codo claro como en las lunas de `fundamentos.ipynb`? ¿Dónde se dispara?
- Según la tabla, ¿qué pasa con el % de clientes que no son núcleo al subir `eps`?
- ¿En qué **rango** de `eps` buscarías?

RELLENAR CON TUS COMENTARIOS

---
# 2️⃣ Grilla `eps` × `min_samples`: el trade-off del ruido (20 min)

Probamos todas las combinaciones de un rango de `eps` (alrededor de lo que vimos) y varios `min_samples`. `grilla_dbscan` usa `evaluar_clustering`, así que la silhouette se calcula **sin el ruido**.

In [ ]:
# 1. tabla_grilla = grilla_dbscan(X_scaled, eps_values=[0.3, 0.4, 0.45, 0.5, 0.55, 0.6, 0.7, 0.8],
#                                 min_samples_values=[4, 6, 8, 10])
# 2. Muestra tabla_grilla
# ✍️ Tu código aquí


Con 32 filas es difícil ver el patrón. Lo acomodamos como **tablas de doble entrada** (`eps` en las filas, `min_samples` en las columnas):

In [ ]:
# Para cada métrica ('n_clusters', 'pct_ruido', 'silhouette') imprime
# tabla_grilla.pivot(index='eps', columns='min_samples', values=metrica)
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Qué pasa con el número de clusters cuando `eps` es grande? ¿Y con el ruido cuando `eps` es chico?
- Busca la silhouette **más alta** de la grilla. ¿Qué % de ruido tiene? ¿Usarías ese modelo para el centro comercial?
- ¿Por qué crees que la silhouette sube cuando sube el ruido?

RELLENAR CON TUS COMENTARIOS

### Un criterio que combine las dos cosas
Necesitamos una **regla de decisión** antes de mirar el "mejor" número:
1. **`min_samples = 6`** (la regla $2 \times$ dimensiones que fijamos en la Parte 1).
2. **Ruido ≤ 30%**: no podemos dejar fuera de las campañas a más de 1 de cada 3 clientes.
3. **Al menos 3 clusters**: con 1 o 2 grupos no hay segmentación útil.
4. Entre lo que sobrevive, la **mejor silhouette**.

In [ ]:
# Filtra tabla_grilla con el criterio y ordena por silhouette de mayor a menor
# Pista: tabla_grilla.query('min_samples == 6 and pct_ruido <= 30 and n_clusters >= 3')
#        y después .sort_values('silhouette', ascending=False)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuántas combinaciones cumplen el criterio? ¿Cuál elegirías y por qué? Anota tu `eps` elegido.

RELLENAR CON TUS COMENTARIOS

---
# 3️⃣ El modelo elegido (10 min)

Ajustamos DBSCAN con los parámetros elegidos y lo guardamos con su propio nombre (`dbscan_mall`), sin pisar `kmeans_3d`.

In [ ]:
# 1. dbscan_mall = DBSCAN(eps=<tu eps>, min_samples=6).fit(X_scaled)
# 2. labels_dbscan = dbscan_mall.labels_
# 3. comparacion.append(evaluar_clustering('DBSCAN', X_scaled, labels_dbscan))
# 4. pd.DataFrame(comparacion)
# ✍️ Tu código aquí


In [ ]:
# Grafica los clusters en 3D y en 2D (ingreso vs gasto)
# Pista: plot_clusters_3d(df, 'edad', 'ingreso_anual', 'gasto', labels_dbscan)
#        plot_clusters(df, 'ingreso_anual', 'gasto', labels_dbscan)
# ✍️ Tu código aquí


✍️ **¿Qué observas?** ¿Cuántos clusters encontró DBSCAN? ¿Qué % de ruido? ¿En qué zonas de la gráfica de ingreso vs gasto se concentra el ruido (gris)?

RELLENAR CON TUS COMENTARIOS

---
# 4️⃣ ¿Quiénes son el ruido? (15 min)

"Ruido" es el nombre técnico, pero en negocio puede ser tres cosas muy distintas:
- **Errores de captura** (edad 250, ingreso negativo) → se corrigen o se eliminan.
- **Atípicos interesantes** (clientes únicos) → atención **personalizada**.
- **Un segmento disperso**: clientes parecidos entre sí pero no tan apretados como para que DBSCAN los vea como grupo.

Empecemos por el perfil de cada cluster y del ruido, en unidades originales:

In [ ]:
# Perfil (tamaño y promedios) de cada cluster de DBSCAN y del ruido
# Pista: perfil_clusters(df, labels_dbscan, COLS_MALL)
# ✍️ Tu código aquí


¿De qué segmentos de K-Means vienen los clientes "ruido"? Una tabla cruzada lo dice (filas = DBSCAN, columnas = segmento de K-Means):

In [ ]:
# 1. grupo_dbscan = np.where(labels_dbscan == -1, 'Ruido', 'DBSCAN ' + labels_dbscan.astype(str))
# 2. pd.crosstab(grupo_dbscan, segmento_kmeans, rownames=['DBSCAN'], colnames=['Segmento K-Means'])
# ✍️ Tu código aquí


In [ ]:
# 1. ruido = df[labels_dbscan == -1]
# 2. Cuenta cuántos clientes del ruido tienen ingreso > 70 y gasto < 40, y cuántos ingreso < 40 y gasto < 40
# 3. Revisa en qué cluster quedaron los 8 clientes con mayor ingreso:
#    top_ingreso = df.nlargest(8, 'ingreso_anual').assign(cluster_dbscan=labels_dbscan[df.nlargest(8, 'ingreso_anual').index])
# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Cómo es el cliente "ruido" promedio (ingreso, gasto, edad)? ¿Se parece a algún cluster?
- En la tabla cruzada, ¿de qué segmentos de K-Means vienen la mayoría de los clientes ruido?
- ¿Cuántos son de **ingreso alto y gasto bajo**? ¿En qué cluster quedaron los 8 clientes con mayor ingreso?
- ¿Dirías que el ruido son errores, atípicos interesantes o un segmento disperso? Justifica.

RELLENAR CON TUS COMENTARIOS

---
# 5️⃣ DBSCAN vs K-Means (10 min)

Veamos la tabla de comparación:

In [ ]:
pd.DataFrame(comparacion)

🚩 **Esa comparación no es justa.** La silhouette de K-Means se calculó con los **200** clientes; la de DBSCAN sólo con los clientes que no son ruido (justo los más fáciles). Para comparar en igualdad de condiciones, evaluamos a K-Means **sobre esos mismos clientes**.

Y para saber qué tanto **coinciden** los dos agrupamientos usamos el **ARI** entre ellos (aquí no hay una etiqueta "real": medimos acuerdo entre dos modelos, no acierto).

In [ ]:
# 1. sin_ruido = labels_dbscan != -1
# 2. comparacion.append(evaluar_clustering('K-Means (k=6), sólo clientes no-ruido',
#                                          X_scaled[sin_ruido], kmeans_3d.labels_[sin_ruido]))
# 3. ARI entre ambos con todos los clientes:      adjusted_rand_score(kmeans_3d.labels_, labels_dbscan)
# 4. ARI sólo con los clientes no-ruido (usa [sin_ruido] en ambos)
# 5. pd.DataFrame(comparacion)
from sklearn.metrics import adjusted_rand_score

# ✍️ Tu código aquí


✍️ **¿Qué observas?**
- ¿Cambió la conclusión al comparar la silhouette sobre los mismos clientes?
- ¿Qué tanto coinciden los dos modelos (ARI) con y sin el ruido? ¿Dónde están sus diferencias?

RELLENAR CON TUS COMENTARIOS

---
# 6️⃣ Conclusiones (5 min)

## ❓ Respuesta a la pregunta
> *¿Qué clientes no encajan en ningún segmento, quiénes son y qué hacemos con ellos?*

✍️ Responde con tus resultados: ¿cuántos clientes son ruido y qué perfil tienen? ¿Qué tanto coinciden DBSCAN y K-Means en el resto?

✍️ **Recomendación:** ¿qué modelo usarías para las campañas masivas? ¿Qué harías con los clientes "ruido"? Propón una acción para cada tipo de cliente que encontraste.

RELLENAR CON TUS COMENTARIOS

## 📝 Lo que aprendimos
✍️ Escribe 3 cosas que aprendiste en este laboratorio.

RELLENAR CON TUS COMENTARIOS

## 🚀 Retos opcionales
- Repite el análisis sólo con `ingreso_anual` y `gasto` (2 dimensiones, `min_samples = 4`). ¿Cuánto ruido queda? ¿Aparecen los grupos que se ven a simple vista en la gráfica de ingreso vs gasto?
- Prueba `HDBSCAN(min_cluster_size=10, copy=True)` de `sklearn.cluster` sobre `X_scaled`. ¿Cuántos clusters y cuánto ruido da? ¿Por qué crees que manda tanto a ruido en un dataset tan pequeño?